# Lab | Data Cleaning and Formatting - Solution Notebook

**Dataset**: Insurance company customer data

**Source**: https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

## Setup - Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv")
df.head()

---
# Challenge 1: Data Cleaning and Formatting

## Exercise 1: Cleaning Column Names

**Goal**: Standardize all column names to be:
- Lowercase
- Spaces replaced with underscores `_`
- `st` renamed to `state`

In [ ]:
# First, let's see the current column names
print("BEFORE cleaning:")
print(df.columns.tolist())

In [ ]:
# Step 1: Convert all column names to lowercase
# List comprehension loops through each column name and applies .lower()
df.columns = [col.lower() for col in df.columns]

# Step 2: Replace spaces with underscores
# List comprehension loops through each column name and applies .replace()
df.columns = [col.replace(' ', '_') for col in df.columns]

# Step 3: Rename 'st' to 'state' for clarity
# .rename() takes a dictionary: {'old_name': 'new_name'}
df = df.rename(columns={'st': 'state'})

print("AFTER cleaning:")
print(df.columns.tolist())

**Result**: All column names are now clean, consistent, and easy to work with.

---
## Exercise 2: Cleaning Invalid Values

**Goal**: Fix inconsistent values in multiple columns.

In [ ]:
# First, let's check the current unique values in each column that needs cleaning
columns_to_check = ['gender', 'state', 'education', 'vehicle_class']

for col in columns_to_check:
    print(f"--- {col} ---")
    print(df[col].unique())
    print()

In [ ]:
# Fix 1: Gender column
# Problem: 'F', 'Femal', 'female' should all be 'F'
#          'M', 'Male', 'male' should all be 'M'
# Solution: .replace() maps old values to new values
df['gender'] = df['gender'].replace({
    'Femal': 'F',
    'female': 'F',
    'Male': 'M'
})

print("Gender after cleaning:")
print(df['gender'].value_counts())
print()

In [ ]:
# Fix 2: State column
# Problem: 'AZ' should be 'Arizona', 'Cali' should be 'California', 'WA' should be 'Washington'
df['state'] = df['state'].replace({
    'AZ': 'Arizona',
    'Cali': 'California',
    'WA': 'Washington'
})

print("State after cleaning:")
print(df['state'].value_counts())
print()

In [ ]:
# Fix 3: Education column
# Problem: 'Bachelors' should be 'Bachelor'
df['education'] = df['education'].replace({'Bachelors': 'Bachelor'})

print("Education after cleaning:")
print(df['education'].value_counts())
print()

In [ ]:
# Fix 4: Customer Lifetime Value column
# Problem: Values contain a '%' sign (e.g., '697953.59%') - can't do math on strings
# Solution: .str.replace() removes the '%' character from every value
df['customer_lifetime_value'] = df['customer_lifetime_value'].str.replace('%', '')

print("Customer Lifetime Value after removing %:")
print(df['customer_lifetime_value'].head())
print()

In [ ]:
# Fix 5: Vehicle Class column
# Problem: 'Sports Car', 'Luxury SUV', 'Luxury Car' should all be 'Luxury'
df['vehicle_class'] = df['vehicle_class'].replace({
    'Sports Car': 'Luxury',
    'Luxury SUV': 'Luxury',
    'Luxury Car': 'Luxury'
})

print("Vehicle Class after cleaning:")
print(df['vehicle_class'].value_counts())

---
## Exercise 3: Formatting Data Types

**Goal**: Fix two columns with incorrect data types:
1. `customer_lifetime_value` — should be float (currently string)
2. `number_of_open_complaints` — should be numeric (currently formatted as `1/0/00`)

In [ ]:
# First, check current data types
print("Current data types:")
print(df.dtypes)

In [ ]:
# Fix 1: Convert customer_lifetime_value to float
# We already removed the '%' sign in Exercise 2
# Now we just need to convert the string to a number using .astype(float)
df['customer_lifetime_value'] = df['customer_lifetime_value'].astype(float)

print("customer_lifetime_value type:", df['customer_lifetime_value'].dtype)
print(df['customer_lifetime_value'].head())
print()

In [ ]:
# Fix 2: Fix number_of_open_complaints column
# Problem: Values look like '1/0/00', '1/2/00' etc.
# The middle number is the actual count of complaints
# '1/0/00' → 0 complaints, '1/2/00' → 2 complaints

# First let's see what unique values exist
print("Unique values before fixing:")
print(df['number_of_open_complaints'].unique())

In [ ]:
# Step 1: Split each value by '/' → '1/0/00' becomes ['1', '0', '00']
# Step 2: Take the middle element (index 1) → '0'
# Step 3: Convert to float (using float instead of int because there are NaN values)
df['number_of_open_complaints'] = df['number_of_open_complaints'].str.split('/').str[1].astype(float)

print("Unique values after fixing:")
print(df['number_of_open_complaints'].unique())
print()
print("Data type:", df['number_of_open_complaints'].dtype)

---
## Exercise 4: Dealing with Null Values

**Goal**: Identify null values, decide on a strategy, and handle them.

In [ ]:
# Step 1: Identify null values in each column
print("Null values per column:")
print(df.isnull().sum())
print()
print(f"Total rows: {len(df)}")

In [ ]:
# Step 2: Check what percentage of each column is null
print("Percentage of null values per column:")
print((df.isnull().sum() / len(df) * 100).round(2))

**Strategy Decision**:

- **Rows where ALL values are null** → Drop them (they contain no useful information)
- **Numerical columns with nulls** (`customer_lifetime_value`, `income`, `monthly_premium_auto`, `number_of_open_complaints`, `total_claim_amount`) → Fill with the **mean** (average)
- **Categorical columns with nulls** (`gender`, `state`, `education`, `policy_type`, `vehicle_class`) → Fill with the **mode** (most common value)

**Why mean for numerical?** It preserves the overall distribution of the data.

**Why mode for categorical?** We cannot average text values, so we use the most common one.

In [ ]:
# Step 3a: Drop rows where ALL values are null
df = df.dropna(how='all')
print(f"Rows after dropping fully empty rows: {len(df)}")

In [ ]:
# Step 3b: Fill numerical columns with mean
numerical_cols = ['customer_lifetime_value', 'income', 'monthly_premium_auto',
                  'number_of_open_complaints', 'total_claim_amount']

for col in numerical_cols:
    df[col] = df[col].fillna(df[col].mean())
    
print("Numerical nulls filled with mean.")

In [ ]:
# Step 3c: Fill categorical columns with mode (most frequent value)
categorical_cols = ['gender', 'state', 'education', 'policy_type', 'vehicle_class']

for col in categorical_cols:
    # .mode()[0] gets the most frequent value
    df[col] = df[col].fillna(df[col].mode()[0])
    
print("Categorical nulls filled with mode.")

In [ ]:
# Step 4: Verify no more null values remain
print("Null values after cleaning:")
print(df.isnull().sum())

In [ ]:
# Step 5: Convert all numeric columns to integers (as required by the lab)
# Note: We must do this AFTER filling nulls - integers cannot hold NaN values
for col in numerical_cols:
    df[col] = df[col].astype(int)

print("Data types after converting to integer:")
print(df.dtypes)

---
## Exercise 5: Dealing with Duplicates

**Goal**: Identify and handle duplicate rows.

In [ ]:
# Step 1: Check how many duplicate rows exist
# .duplicated() returns True for rows that are duplicates
# .sum() counts how many True values there are
num_duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {num_duplicates}")

In [ ]:
# Step 2: Drop duplicate rows
# Strategy: Keep the first occurrence of each duplicate, drop the rest
# This is the safest approach - we don't lose any unique data
# keep='first' means we keep the first time a row appears
df = df.drop_duplicates(keep='first')

print(f"Rows after dropping duplicates: {len(df)}")

In [ ]:
# Step 3: Reset the index after dropping rows
# When we drop rows, the index numbers have gaps (e.g., 0, 1, 5, 8...)
# reset_index() renumbers them cleanly from 0
# drop=True means don't add the old index as a new column
df = df.reset_index(drop=True)

print("Index reset. First few rows:")
print(df.head())

In [ ]:
# Step 4: Verify no duplicates remain
print(f"Duplicate rows remaining: {df.duplicated().sum()}")

In [ ]:
# Step 5: Save the cleaned dataset to a new CSV file
# index=False means don't save the row numbers as a column
df.to_csv('insurance_cleaned.csv', index=False)

print("Cleaned dataset saved to 'insurance_cleaned.csv'")
print(f"Final dataset shape: {df.shape}")

---
## Bonus: Challenge 3 - Analyzing Clean and Formatted Data

**Goal**: Find customers with HIGH claim amounts AND LOW customer lifetime value — these are the most costly and least profitable customers.

In [ ]:
# Step 1: Review statistics for both columns
print("Summary statistics for Total Claim Amount and Customer Lifetime Value:")
print(df[['total_claim_amount', 'customer_lifetime_value']].describe())

In [ ]:
# Step 2: Calculate the thresholds
# High claim amount = above the 75th percentile
claim_75th = df['total_claim_amount'].quantile(0.75)

# Low customer lifetime value = below the 25th percentile
clv_25th = df['customer_lifetime_value'].quantile(0.25)

print(f"75th percentile of Total Claim Amount: {claim_75th:.2f}")
print(f"25th percentile of Customer Lifetime Value: {clv_25th:.2f}")
print()
print("We want customers with:")
print(f"  - Total Claim Amount > {claim_75th:.2f} (top 25% claimers)")
print(f"  - Customer Lifetime Value < {clv_25th:.2f} (bottom 25% value)")

In [ ]:
# Step 3: Filter the DataFrame
# & means both conditions must be True
high_risk_df = df[
    (df['total_claim_amount'] > claim_75th) &
    (df['customer_lifetime_value'] < clv_25th)
]

print(f"Number of high-risk customers: {len(high_risk_df)}")
print(f"This represents {len(high_risk_df)/len(df)*100:.1f}% of all customers")

In [ ]:
# Step 4: Summary statistics for the high-risk segment
print("Summary statistics for high-risk customers:")
print(high_risk_df[['total_claim_amount', 'customer_lifetime_value']].describe())

**Business Insight**:

These customers are the most concerning for the insurance company:
- They make **large claims** (costing the company money)
- They have **low lifetime value** (they don't generate much revenue)

This segment should be reviewed for:
- Premium price adjustments
- Policy renewal decisions
- Fraud investigation
- Targeted retention or risk mitigation strategies